In [13]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from typing import cast, Iterable
from anthropic.types import MessageParam
from typing import cast, Iterable


# Lessons 3: Making a Request

In [16]:
# Making Your First Request

client = Anthropic()
model = "claude-sonnet-4-5"

message = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = cast(Iterable[MessageParam], cast(object, [
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]))
)

In [15]:
message.content[0].text

'Quantum computing is a type of computation that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.'

# Lesson 4: Multi-Turn conversations

In [17]:
# Building Helper Functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model = model,
        max_tokens = 1000,
        messages = messages,
    )
    return message.content[0].text

In [28]:
# Putting it all together
messages = []

# First Turn
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant(messages, answer)

# Second Turn
add_user_message(messages, "Write another sentence")
final_answer = chat(messages)
add_assistant(messages, final_answer)

In [29]:
for msg in messages:
    print(f"{msg['role'].capitalize()}: {msg['content']}")

User: Define quantum computing in one sentence
Assistant: Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain problems exponentially faster than classical computers.
User: Write another sentence
Assistant: Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits which are either 0 or 1.


# Lesson 5: System Prompts

In [48]:
messages = [
    {
        "role": "user",
        "content": "How do I solve 5x + 2 = 3 for x?"
    }
]

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

# Building a Flexible Chat Function

def chat(messages, system = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    if not message.content:
         return f"Error: No content returned. Stop reason: {message.stop_reason}"

    return message.content[0].text

In [49]:
print("--- TESTING WITHOUT SYSTEM PROMPT ---")
answer_without = chat(messages)
print(answer_without)

--- TESTING WITHOUT SYSTEM PROMPT ---
# Solving 5x + 2 = 3

To solve for x, isolate the variable by reversing the operations:

**Step 1:** Subtract 2 from both sides
- 5x + 2 - 2 = 3 - 2
- 5x = 1

**Step 2:** Divide both sides by 5
- 5x/5 = 1/5
- x = 1/5

**Answer: x = 1/5** (or x = 0.2)

**Check:** 5(1/5) + 2 = 1 + 2 = 3 ✓


In [50]:
print("\n--- TESTING WITH SYSTEM PROMPT ---")
answer_with = chat(messages, system=system_prompt)
print(answer_with)


--- TESTING WITH SYSTEM PROMPT ---
Good question! Let's work through this together step by step.

First, I want you to think about what we're trying to do here. We want to get **x by itself** on one side of the equation.

Looking at the left side, we have **5x + 2**. 

**What do you think should be our first step to start isolating x?** 

Hint: What operation could we perform on both sides to get rid of the "+ 2"?
